# Day 6 — Capstone: Enterprise RAG Chatbot

---

Time to ship. Today we combine **everything** from Section 5 & 6 into a real, deployable **Enterprise RAG Chatbot** with:

- Multi-format ingestion (PDFs, DOCX, web)
- Reranked retrieval
- Streaming answers with inline citations
- Refusal when confidence is low
- Prompt-injection filter
- JWT auth (reused from Section 2)
- Per-user token & cost logging

The whole thing is one FastAPI app. ~300 lines.


## Architecture

```
   [User]
     │  POST /ingest   (with JWT)
     │  POST /ask      (with JWT)
     ▼
   [FastAPI]  ── auth ──> [JWT verify]  (Section 2 pattern)
     │
     ├─► /ingest ──► load() ──► clean ──► chunk ──► embed ──► Chroma
     │
     └─► /ask
             │
             ├─► injection filter (Day 5)
             ├─► expand_query    (Day 3, optional)
             ├─► retrieve top-20 from Chroma
             ├─► rerank -> top-5 (Day 3)
             ├─► distance threshold check (Day 5)
             ├─► build_prompt with citations (Day 4)
             └─► stream from Together AI (Section 4 Day 6)
                    │
                    └─► log tokens per user (SQLite)
```

Every arrow is code you've already written. Today is about wiring, not new concepts.


## Endpoints


### `POST /ingest`

**Body:**
```json
{"source": "./docs/handbook.pdf"}
```
or
```json
{"source": "https://en.wikipedia.org/wiki/RAG"}
```

**Response:**
```json
{"chunks_added": 42, "source": "handbook.pdf"}
```

Requires a valid JWT header. The uploader's `user_id` gets tagged as metadata on every chunk — enables per-user access control later.


### `POST /ask` (streaming)

**Body:**
```json
{"question": "What's our refund policy?", "top_k": 5}
```

**Response:** `text/event-stream` — tokens stream in as the LLM generates them, ending with a JSON blob of citations.

**Behavior:**
- Injection-detected → returns `"I can't help with that."` (logged).
- Low retrieval confidence → returns `"I don't know."` (no LLM call, saves cost).
- Otherwise → streams the answer with `[1]`, `[2]` citations.


### `GET /usage` — per-user stats
```json
{"user": "alice", "questions": 47, "input_tokens": 12043, "output_tokens": 3812, "est_cost_usd": 0.0121}
```


## The full app — walk through the file

Open `main.py` in this folder. It's organized top-to-bottom:

1. **Imports & setup** — Chroma persistent client, embedder, reranker, Together client, SQLite for logs
2. **Loader registry** — from Day 2
3. **Chunking + injection filter** — from S5 D5 + Day 5
4. **Auth helpers** — reused from Section 2 JWT
5. **Endpoints** — `/ingest`, `/ask`, `/usage`
6. **Streaming generator** for `/ask`

**Try the full flow:**

```bash
# 1. Start it
uvicorn main:app --reload

# 2. Get a token (the /login is a stub — replace with your real user table)
curl -X POST http://localhost:8000/login \
     -H "Content-Type: application/json" \
     -d '{"username":"alice","password":"pw"}'
# → {"token": "eyJ..."}

# 3. Ingest a doc
curl -X POST http://localhost:8000/ingest \
     -H "Authorization: Bearer eyJ..." \
     -H "Content-Type: application/json" \
     -d '{"source":"./sample.pdf"}'

# 4. Ask a question (streaming)
curl -N -X POST http://localhost:8000/ask \
     -H "Authorization: Bearer eyJ..." \
     -H "Content-Type: application/json" \
     -d '{"question":"What is the refund policy?"}'

# 5. Check usage
curl http://localhost:8000/usage -H "Authorization: Bearer eyJ..."
```

Or use the Swagger UI at http://localhost:8000/docs


## Design decisions worth noting

- **Persistent Chroma** — data survives restarts. Good enough for single-server production; swap to Pinecone / pgvector when you scale.
- **Reranker loaded once at startup** — the model is ~90 MB. Loading it per-request would be catastrophic.
- **Streaming, not buffered** — users see the first token in ~1s; buffered answers wait 5–10s. Huge UX win.
- **Cost tracking in SQLite** — cheap, atomic, easy to query. Grafana on top of this table gives you a dashboard for free.
- **JWT verify as a dependency** — every endpoint that needs auth just adds `user = Depends(current_user)`. Section 2 pattern.


## What you should extend (portfolio-worthy)

Pick one and add it before you call this done:

1. **Per-user KBs.** Filter Chroma by `{"user_id": user_id}` so users only search their own docs.
2. **Daily spend cap.** If a user exceeds $1/day of token cost, return `429 Too Many Requests`.
3. **Simple HTML frontend.** One page with a search box and streaming answer via `EventSource`.
4. **A `/feedback` endpoint** — users mark answers as good/bad. Save it. That's your future eval set.

Any of these turns the capstone into a **real portfolio project**.


## What you've built across Sections 5 & 6

Two weeks of work, one production-quality RAG:

- Chunks and embeds arbitrary documents (Section 5)
- Semantic + hybrid search over a persistent vector DB (Section 5)
- Reranks candidates with a cross-encoder (Section 6)
- Refuses when confidence is low, defends against prompt injection (Section 6)
- Streams cited answers over an authenticated HTTP API (Section 6)
- Logs cost per user (Section 6)

**This is the exact stack behind Notion AI, Perplexity's basic mode, most enterprise support bots, and every "chat with your PDF" app you've ever used.**

Great work. See you in Section 7 — AI Agents.
